In [ ]:
# Figure plotting: Encroachment and retreat trends

In [ ]:
# ==============================================================
# Sub-region woody change trends — 5-year & hydrological periods
# Hydrological periods plotted on a LINEAR time axis
# ==============================================================

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------
# CONFIG
# -----------------------
STEP3_CSV = "/home/jovyan/DEA products paper/Results/step 3_v3/tables/wofs_mask_1988_2023_5yr_plusEpochs_20251007_084143.csv"

SR_AREA_UNIQUE = {
    "SR1": 4576.62,
    "SR2": 7843.29,
    "SR3": 8808.87,
    "SR4": 7065.60,
    "SR5": 3769.56,
}

SR_COLORS = {
    "SR1": "#1b9e77",
    "SR2": "#d95f02",
    "SR3": "#7570b3",
    "SR4": "#e7298a",
    "SR5": "#66a61e",
}

SR_MARKERS = {
    "SR1": "o",
    "SR2": "s",
    "SR3": "^",
    "SR4": "D",
    "SR5": "v",
}

# Hydrological periods
EPOCHS = {
    "1988-1993": "Wet Baseline",
    "1994-2000": "Early Dry",
    "2001-2009": "Millennium Drought",
    "2010-2012": "Post-Drought Floods",
    "2013-2017": "Moderate Dry",
    "2018-2019": "Severe Drought",
    "2020-2022": "La Niña Floods",
}
EPOCH_ORDER = list(EPOCHS.values())

# Canonical 5-year bins
FIVE_YEAR_ORDER = [
    "1988-1993",
    "1993-1998",
    "1998-2003",
    "2003-2008",
    "2008-2013",
    "2013-2018",
    "2018-2023",
]

# -----------------------
# HELPERS
# -----------------------
def extract_period_base(p):
    m = re.match(r"\s*(\d{4})\s*-\s*(\d{4})", str(p))
    return f"{m.group(1)}-{m.group(2)}" if m else None

def period_midpoint(p):
    m = re.match(r"\s*(\d{4})\s*-\s*(\d{4})", str(p))
    if m:
        start = int(m.group(1))
        end = int(m.group(2))
        return (start + end) / 2.0
    return np.nan

# -----------------------
# LOAD & DEDUP
# -----------------------
df = pd.read_csv(STEP3_CSV).fillna(0)

df["PeriodBase"] = df["Period"].apply(extract_period_base)
df = df[~df["PeriodBase"].isna()].copy()
df["TimeMid"] = df["PeriodBase"].apply(period_midpoint)
df["Epoch"] = df["PeriodBase"].map(EPOCHS)

dedup_keys = ["WetlandID", "SubRegion", "PeriodBase"]
df = df.sort_values(dedup_keys).drop_duplicates(subset=dedup_keys, keep="first")

# -----------------------
# AGGREGATE 5-YEAR
# -----------------------
df5 = df[df["PeriodBase"].isin(FIVE_YEAR_ORDER)].copy()

g5 = (
    df5.groupby(["SubRegion", "PeriodBase"])[
        ["Enc_Any_to_Woody_ha_after", "Ret_Any_to_NonWoody_ha_after"]
    ]
    .sum()
    .reset_index()
)

g5["Enc_pct_5yr"] = g5.apply(
    lambda r: 100.0 * r["Enc_Any_to_Woody_ha_after"] / SR_AREA_UNIQUE[r["SubRegion"]],
    axis=1
)
g5["Ret_pct_5yr"] = g5.apply(
    lambda r: 100.0 * r["Ret_Any_to_NonWoody_ha_after"] / SR_AREA_UNIQUE[r["SubRegion"]],
    axis=1
)

g5["PeriodBase"] = pd.Categorical(g5["PeriodBase"], categories=FIVE_YEAR_ORDER, ordered=True)

enc_5yr = g5[["SubRegion", "PeriodBase", "Enc_pct_5yr"]].sort_values(["PeriodBase", "SubRegion"])
ret_5yr = g5[["SubRegion", "PeriodBase", "Ret_pct_5yr"]].sort_values(["PeriodBase", "SubRegion"])

# -----------------------
# AGGREGATE HYDROLOGICAL PERIODS
# -----------------------
ge = (
    df.dropna(subset=["Epoch"])
    .groupby(["SubRegion", "PeriodBase", "TimeMid", "Epoch"])[
        ["Enc_Any_to_Woody_ha_after", "Ret_Any_to_NonWoody_ha_after"]
    ]
    .sum()
    .reset_index()
)

ge["Enc_pct"] = ge.apply(
    lambda r: 100.0 * r["Enc_Any_to_Woody_ha_after"] / SR_AREA_UNIQUE[r["SubRegion"]],
    axis=1
)
ge["Ret_pct"] = ge.apply(
    lambda r: 100.0 * r["Ret_Any_to_NonWoody_ha_after"] / SR_AREA_UNIQUE[r["SubRegion"]],
    axis=1
)

epoch_rank = {name: i for i, name in enumerate(EPOCH_ORDER)}
ge["EpochRank"] = ge["Epoch"].map(epoch_rank)

enc_epoch = ge[["SubRegion", "PeriodBase", "TimeMid", "Epoch", "EpochRank", "Enc_pct"]].sort_values(
    ["TimeMid", "SubRegion"]
)
ret_epoch = ge[["SubRegion", "PeriodBase", "TimeMid", "Epoch", "EpochRank", "Ret_pct"]].sort_values(
    ["TimeMid", "SubRegion"]
)

# -----------------------
# STYLE SETTINGS
# -----------------------
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth": 1.0,
    "figure.dpi": 300,
})

ymax = np.ceil(
    max(
        enc_5yr["Enc_pct_5yr"].max(),
        ret_5yr["Ret_pct_5yr"].max(),
        enc_epoch["Enc_pct"].max(),
        ret_epoch["Ret_pct"].max(),
    ) / 5.0
) * 5.0

def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.grid(True, linestyle=":", alpha=0.15)
    ax.xaxis.grid(False)

# -----------------------
# FIGURE 3 — 5-Year Intervals
# -----------------------
fig3, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

for sr in sorted(enc_5yr["SubRegion"].unique()):
    s = enc_5yr[enc_5yr["SubRegion"] == sr]
    ax1.plot(
        s["PeriodBase"], s["Enc_pct_5yr"],
        marker=SR_MARKERS[sr], markersize=7, linewidth=2,
        color=SR_COLORS[sr], label=sr
    )

for sr in sorted(ret_5yr["SubRegion"].unique()):
    s = ret_5yr[ret_5yr["SubRegion"] == sr]
    ax2.plot(
        s["PeriodBase"], s["Ret_pct_5yr"],
        marker=SR_MARKERS[sr], markersize=7, linewidth=2,
        linestyle="--", color=SR_COLORS[sr], label=sr
    )

for ax in [ax1, ax2]:
    ax.set_ylim(0, ymax)
    clean_axes(ax)

ax1.set_title("(a) Encroachment", loc="left", fontweight="semibold", fontsize=12)
ax2.set_title("(b) Retreat", loc="left", fontweight="semibold", fontsize=12)
ax1.set_ylabel("Encroachment (% of wetland area)")
ax2.set_ylabel("Retreat (% of wetland area)")
ax2.set_xticklabels(FIVE_YEAR_ORDER, rotation=35, ha="right")

handles, labels = ax1.get_legend_handles_labels()
fig3.legend(
    handles, labels,
    loc="upper center", ncol=5, frameon=False,
    bbox_to_anchor=(0.5, 1.05)
)

plt.tight_layout()
fig3.savefig("Fig3_Combined_5yr_Pure.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

# -----------------------
# FIGURE 4 — Hydrological Periods (LINEAR TIME SCALE)
# -----------------------
fig4, (ax3, ax4) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

for sr in sorted(enc_epoch["SubRegion"].unique()):
    s = enc_epoch[enc_epoch["SubRegion"] == sr].sort_values("TimeMid")
    ax3.plot(
        s["TimeMid"], s["Enc_pct"],
        marker=SR_MARKERS[sr], markersize=7, linewidth=2,
        color=SR_COLORS[sr], label=sr
    )

for sr in sorted(ret_epoch["SubRegion"].unique()):
    s = ret_epoch[ret_epoch["SubRegion"] == sr].sort_values("TimeMid")
    ax4.plot(
        s["TimeMid"], s["Ret_pct"],
        marker=SR_MARKERS[sr], markersize=7, linewidth=2,
        linestyle="--", color=SR_COLORS[sr], label=sr
    )

for ax in [ax3, ax4]:
    ax.set_ylim(0, ymax)
    clean_axes(ax)

ax3.set_title("(a) Encroachment", loc="left", fontweight="semibold", fontsize=12)
ax4.set_title("(b) Retreat", loc="left", fontweight="semibold", fontsize=12)
ax3.set_ylabel("Encroachment (% of wetland area)")
ax4.set_ylabel("Retreat (% of wetland area)")

label_map = {
    "Wet Baseline": "Wet\nBaseline\n(1988–1993)",
    "Early Dry": "Early Dry\n(1994–2000)",
    "Millennium Drought": "Millennium Drought\n(2001–2009)",
    "Post-Drought Floods": "Post-Drought\nFloods\n(2010–2012)",
    "Moderate Dry": "Moderate Dry\n(2013–2017)",
    "Severe Drought": "Severe\nDrought\n(2018–2019)",
    "La Niña Floods": "La Niña\nFloods\n(2020–2022)",
}

epoch_tick_positions = [period_midpoint(p) for p in EPOCHS.keys()]
epoch_tick_labels = [label_map[label] for label in EPOCHS.values()]

ax4.set_xticks(epoch_tick_positions)
ax4.set_xticklabels(
    epoch_tick_labels,
    rotation=0,
    ha="center",
    fontsize=8.5,
    linespacing=0.9
)
ax4.tick_params(axis="x", pad=6)

xmin = min(epoch_tick_positions) - 1.0
xmax = max(epoch_tick_positions) + 1.0
ax3.set_xlim(xmin, xmax)

handles, labels = ax3.get_legend_handles_labels()
fig4.legend(
    handles, labels,
    loc="upper center", ncol=5, frameon=False,
    bbox_to_anchor=(0.5, 1.05)
)

plt.subplots_adjust(left=0.10, right=0.95, top=0.90, bottom=0.22, hspace=0.12)
fig4.savefig("Fig4_Combined_Periods_LinearTime.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved:")
print("  Fig3_Combined_5yr_Pure.png")
print("  Fig4_Combined_Periods_LinearTime.png")